# Laboratorio 3 — Comparar y seleccionar un modelo

En el **Laboratorio 2** entrenamos, para cada tarea, dos candidatos:

- Regresión: `LinearRegression` y `RandomForestRegressor`.
- Clasificación: `LogisticRegression` y `RandomForestClassifier`.

Cada uno quedó registrado como un `run` independiente en MLflow. Pero
todavía no habíamos hecho la pregunta que importa: **¿cuál de los dos
deberíamos usar, y con qué criterio decidimos eso?**

Este laboratorio corresponde a la idea de la Unidad 4 (Capítulo 3)
*"¿Cuál modelo debemos desplegar?"*: no basta con mirar una métrica y
quedarse con el número más alto. Hay que compararlo contra algo, y ese
"algo" es un **baseline ingenuo** — la respuesta trivial que no necesita
haber aprendido nada.

In [1]:
from academic_analysis.compare import (
    select_classification_model,
    select_regression_model,
)

DATASET_ID = "ING-20260910-101728"

## 1. Regresión: ¿`LinearRegression` o `RandomForestRegressor`?

El baseline aquí es el modelo más simple posible: predecir siempre la nota
**promedio**. Cualquier modelo real tiene que producir un RMSE menor que
ese baseline para justificar su existencia.

In [2]:
regression_selection = select_regression_model(DATASET_ID)
regression_selection

{'criteria': 'Menor RMSE entre los candidatos que superan el baseline (predecir siempre el promedio de entrenamiento).',
 'baseline': {'description': 'Predecir siempre la nota promedio.',
  'rmse': 0.8693518928235887},
 'candidates': [{'model_type': 'linear',
   'run_id': 'a40fd05d1c6d4e7297c88befa36912ef',
   'rmse': 0.8321884656826379,
   'r2': 0.08351088462559308,
   'passes_gate': True},
  {'model_type': 'random_forest',
   'run_id': '524091f52e024cbba5b90797a73b44c5',
   'rmse': 0.8342205512299212,
   'r2': 0.07902954848385213,
   'passes_gate': True}],
 'selected_model_type': 'linear',
 'selected_run_id': 'a40fd05d1c6d4e7297c88befa36912ef',
 'evaluated_at': '2026-09-25T17:03:51.612549-05:00'}

**Lectura**: los dos candidatos superan el baseline (RMSE menor que
predecir siempre el promedio), aunque por poco. Entre ellos,
`LinearRegression` queda ligeramente mejor. En este caso, el modelo más
simple gana — un buen recordatorio de que "más complejo" no es sinónimo
de "mejor": con estas variables (asistencia, curso, semestre) no hay
suficiente señal no lineal para que un Random Forest saque ventaja, y en
cambio sí paga el costo de ser más difícil de explicar.

## 2. Clasificación: ¿`LogisticRegression` o `RandomForestClassifier`?

El baseline aquí es predecir siempre la clase mayoritaria ("aprueba"),
que ya vimos en el Laboratorio 2 que ronda el 78%.

In [3]:
classification_selection = select_classification_model(DATASET_ID)
classification_selection

{'criteria': 'Mayor F1 entre los candidatos que superan el baseline (predecir siempre la clase mayoritaria de entrenamiento).',
 'baseline': {'description': 'Predecir siempre la clase mayoritaria (aprueba).',
  'accuracy': 0.7848932676518884},
 'candidates': [{'model_type': 'logistic',
   'run_id': '69b3a90cebf54052b2ca4504a5dcb884',
   'accuracy': 0.7848932676518884,
   'f1': 0.8794848206071757,
   'passes_gate': False},
  {'model_type': 'random_forest',
   'run_id': '1ddbff78a5a343d7914271bc93fceac8',
   'accuracy': 0.7832512315270936,
   'f1': 0.8784530386740331,
   'passes_gate': False}],
 'selected_model_type': None,
 'selected_run_id': None,
 'evaluated_at': '2026-09-25T17:03:51.648841-05:00'}

**Lectura — este es el resultado más importante del laboratorio**:
**ningún candidato supera el baseline.** Los dos tienen una *accuracy*
prácticamente igual (o incluso menor) que la de responder siempre
"aprueba" sin mirar ningún dato.

`select_classification_model` por diseño **no selecciona ningún modelo**
en este caso. Eso no es un error del código: es exactamente el
comportamiento que buscábamos. La Unidad 4 lo resume así:

> *"No podemos garantizar que nunca ocurrirán problemas, pero podemos
> diseñar qué debe ocurrir cuando aparezcan."*

Promover un modelo que no aporta nada por encima de adivinar sería peor
que no tener modelo: daría una falsa sensación de tener algo funcionando.
Lo correcto, con la información que tenemos hoy, es **no desplegar
todavía** el modelo de clasificación y volver a la mesa de trabajo: la
asistencia, el curso y el semestre solos no alcanzan para predecir la
aprobación mejor que el azar de la clase mayoritaria.

## 3. ¿Qué se guarda de esta comparación?

`select_regression_model` y `select_classification_model` no solo
imprimen un resultado: también lo dejan escrito en
`metadata/<dataset_id>.json`, bajo una nueva clave `selection`, con el
criterio usado, el baseline, cada candidato evaluado y cuál (si alguno)
quedó seleccionado. Así la decisión de qué modelo usar queda tan
trazable como el propio entrenamiento — no vive solo en la cabeza de
quien ejecutó el notebook.

In [4]:
import json

from academic_analysis.train import METADATA_DIR

manifest = json.loads(
    (METADATA_DIR / f"{DATASET_ID}.json").read_text(encoding="utf-8")
)
manifest["selection"]

{'regression': {'criteria': 'Menor RMSE entre los candidatos que superan el baseline (predecir siempre el promedio de entrenamiento).',
  'baseline': {'description': 'Predecir siempre la nota promedio.',
   'rmse': 0.8693518928235887},
  'candidates': [{'model_type': 'linear',
    'run_id': 'a40fd05d1c6d4e7297c88befa36912ef',
    'rmse': 0.8321884656826379,
    'r2': 0.08351088462559308,
    'passes_gate': True},
   {'model_type': 'random_forest',
    'run_id': '524091f52e024cbba5b90797a73b44c5',
    'rmse': 0.8342205512299212,
    'r2': 0.07902954848385213,
    'passes_gate': True}],
  'selected_model_type': 'linear',
  'selected_run_id': 'a40fd05d1c6d4e7297c88befa36912ef',
  'evaluated_at': '2026-09-25T17:03:51.612549-05:00'},
 'classification': {'criteria': 'Mayor F1 entre los candidatos que superan el baseline (predecir siempre la clase mayoritaria de entrenamiento).',
  'baseline': {'description': 'Predecir siempre la clase mayoritaria (aprueba).',
   'accuracy': 0.784893267651888

## 4. Registrar los modelos (Model Registry)

Ya sabemos qué modelo de regresión seleccionar y sabemos que, por ahora,
ningún modelo de clasificación merece usarse. Falta un último paso para que
esa decisión no viva solo en este notebook: **registrarla formalmente**.

La Unidad 4 (Capítulo 3) lo resume así:

> *"Un Model Registry proporciona un mecanismo para organizar modelos y sus
> versiones [...] Así el modelo deja de ser simplemente un archivo perdido
> en una carpeta. Se convierte en un artefacto gestionado dentro del ciclo
> de vida."*

Aquí usamos dos estados, con MLflow *aliases*:

- **`candidato`**: superó el baseline (regresión, `linear`).
- **`archivado`**: no lo superó, pero queda registrado igual — con la
  razón exacta del rechazo — en vez de simplemente desaparecer
  (clasificación, `logistic`).

In [5]:
from academic_analysis.registry import (
    register_regression_model,
    register_rejected_classification_model,
)

regression_entry = register_regression_model(DATASET_ID)
regression_entry

Registered model 'nota_curso_regresion' already exists. Creating a new version of this model...
2026/09/25 17:03:52 WARNING mlflow.tracking._model_registry.fluent: Run with id a40fd05d1c6d4e7297c88befa36912ef has no artifacts at artifact path 'model', registering model based on models:/m-3737d3870f1b4efb9cdc31b9f4e85cc7 instead


Created version '2' of model 'nota_curso_regresion'.


{'registered_model_name': 'nota_curso_regresion',
 'version': 2,
 'alias': 'candidato',
 'run_id': 'a40fd05d1c6d4e7297c88befa36912ef',
 'model_type': 'linear',
 'status': 'candidato',
 'registered_at': '2026-09-25T17:03:52.954994-05:00'}

In [6]:
classification_entry = register_rejected_classification_model(DATASET_ID)
classification_entry

Registered model 'aprobacion_clasificacion' already exists. Creating a new version of this model...


2026/09/25 17:03:53 WARNING mlflow.tracking._model_registry.fluent: Run with id 69b3a90cebf54052b2ca4504a5dcb884 has no artifacts at artifact path 'model', registering model based on models:/m-572b348d44d74f30917481581edceeef instead


Created version '2' of model 'aprobacion_clasificacion'.


{'registered_model_name': 'aprobacion_clasificacion',
 'version': 2,
 'alias': 'archivado',
 'run_id': '69b3a90cebf54052b2ca4504a5dcb884',
 'model_type': 'logistic',
 'status': 'archivado',
 'rejection_reason': 'No supera el baseline ingenuo (accuracy 0.7849 vs. baseline 0.7849).',
 'registered_at': '2026-09-25T17:03:53.135890-05:00'}

**Lectura**: ambos modelos quedan en el Model Registry de MLflow, cada
uno con su alias, sus métricas y —en el caso del rechazado— la razón
explícita del rechazo como *tag* de la versión. Cualquier persona (o
cualquier servicio) puede pedir después *"dame la versión con alias
`candidato` de `nota_curso_regresion`"* sin tener que saber el `run_id`
exacto ni en qué notebook se entrenó.

Esto también queda resumido en `metadata/<dataset_id>.json`, bajo
`registry.regression` y `registry.classification` — el mismo patrón de
trazabilidad que ya veníamos usando desde la ingesta.

In [7]:
from academic_analysis.train import MLFLOW_TRACKING_URI

print(f"uv run mlflow ui --backend-store-uri {MLFLOW_TRACKING_URI}")
print("En la pestaña 'Models' de la interfaz de MLflow se ven ambos modelos registrados.")

uv run mlflow ui --backend-store-uri sqlite:///C:\Users\000010478\Downloads\analisis_datos_academicos\mlflow.db
En la pestaña 'Models' de la interfaz de MLflow se ven ambos modelos registrados.


## 5. Próximos pasos (aún por definir)

Con el modelo de regresión ya registrado como `candidato`, el siguiente
paso natural (Unidad 4, Capítulo 4) es dejar de pensar en "un modelo que
alguien carga en un notebook" y empezar a pensar en "un modelo que un
servicio puede usar":

- Cargar la versión con alias `candidato` de `nota_curso_regresion` desde
  un servicio **FastAPI**.
- Empaquetar ese servicio con **Docker**, para que no dependa del
  computador de quien lo construyó.
- Exponerlo temporalmente con un **túnel de Cloudflare**, para poder
  mostrarlo sin dejar infraestructura corriendo de forma permanente.

El modelo de clasificación, archivado, se queda fuera de esta etapa: no
tendría sentido desplegar un servicio que en la práctica no hace mejor
que adivinar.